# Notebook 02: Concurrencia, Asincronía y asyncio

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sonder-art/fdd_p26/blob/main/clase/16_computo/code/02_concurrencia_asyncio.ipynb)

**Módulo 16 — Clase 2**

Este notebook acompaña los archivos `03_concurrencia_y_asincronia.md` y `04a_asyncio_fundamentos.md`.

Secciones **** se trabajan durante la sesión.  
Secciones **** se completan después.

---

In [1]:
import asyncio
import time
import threading
import os
import sys

print(f'Python {sys.version}')
print(f'asyncio version: {asyncio.__version__ if hasattr(asyncio, "__version__") else "built-in"}')

# Jupyter ya tiene un event loop corriendo — podemos usar await directamente en las celdas
# Si usas un script .py, necesitas asyncio.run(main())

Python 3.13.12 (tags/v3.13.12:1cbe481, Feb  3 2026, 18:22:25) [MSC v.1944 64 bit (AMD64)]
asyncio version: built-in


## Sección 1: await secuencial vs asyncio.gather — la diferencia central

**Qué vamos a ver:** la diferencia entre M2 (await secuencial) y M4 (gather) es literalmente una línea de código. Los tiempos medidos hacen que la diferencia sea imposible de ignorar.

**Predicción del modelo:**
- M2 (await en secuencia): `T_total = N × T_tarea` — cada usuario espera a que el anterior termine
- M4 (gather): `T_total ≈ T_tarea_más_lenta` — todos los usuarios esperan *al mismo tiempo*

Con N=5 tareas de 1.0s cada una:
- M2 debería tardar: **5.0s**
- M4 debería tardar: **~1.0s**

Corre las celdas y verifica. ¿Coincide con la predicción? ¿El speedup es exactamente 5×, o hay overhead?

> Referencia: `04a_asyncio_fundamentos.md` — sección "asyncio.gather — M4 en una línea"

In [2]:
# Tarea simulada con I/O-bound: espera τ segundos
async def tarea_io(nombre: str, duracion: float) -> str:
    # exec(τᵢ): inicializar
    inicio = time.perf_counter()
    # wait(τᵢ): simula I/O (llamada a API, lectura de BD, etc.)
    await asyncio.sleep(duracion)
    # exec(τᵢ): procesar resultado
    elapsed = time.perf_counter() - inicio
    return f'{nombre}: {elapsed:.2f}s'

DURACION = 1.0  # cada tarea tarda 1s de I/O
N_TAREAS = 5

# --- M2: await secuencial (esperas NO explotadas) ---
t0 = time.perf_counter()
resultados_m2 = []
for i in range(N_TAREAS):
    r = await tarea_io(f'τ{i+1}', DURACION)
    resultados_m2.append(r)
t_m2 = time.perf_counter() - t0

print(f'=== M2: await secuencial ===')
for r in resultados_m2:
    print(f'  {r}')
print(f'Tiempo total M2: {t_m2:.2f}s  (esperado: {N_TAREAS * DURACION:.1f}s = N×T)')
print()

=== M2: await secuencial ===
  τ1: 1.00s
  τ2: 1.01s
  τ3: 1.01s
  τ4: 1.01s
  τ5: 1.01s
Tiempo total M2: 5.04s  (esperado: 5.0s = N×T)



In [3]:
# --- M4: asyncio.gather (esperas SÍ explotadas) ---
t0 = time.perf_counter()
resultados_m4 = await asyncio.gather(
    *[tarea_io(f'τ{i+1}', DURACION) for i in range(N_TAREAS)]
)
t_m4 = time.perf_counter() - t0

print(f'=== M4: asyncio.gather ===')
for r in resultados_m4:
    print(f'  {r}')
print(f'Tiempo total M4: {t_m4:.2f}s  (esperado: ~{DURACION:.1f}s = T_max)')
print()
print(f'Speedup M4/M2: {t_m2/t_m4:.1f}x')
print()
print(f'Conclusión: gather explota las esperas — exec(τⱼ) ∩ wait(τᵢ) ≠ ∅')
print(f'Las {N_TAREAS} tareas de {DURACION}s corren en ~{DURACION}s en lugar de {N_TAREAS*DURACION}s')

=== M4: asyncio.gather ===
  τ1: 1.01s
  τ2: 1.01s
  τ3: 1.01s
  τ4: 1.01s
  τ5: 1.01s
Tiempo total M4: 1.01s  (esperado: ~1.0s = T_max)

Speedup M4/M2: 5.0x

Conclusión: gather explota las esperas — exec(τⱼ) ∩ wait(τᵢ) ≠ ∅
Las 5 tareas de 1.0s corren en ~1.0s en lugar de 5.0s


## Sección 2: Traza del event loop con asyncio debug mode

**Qué vamos a ver:** asyncio tiene un modo de depuración que emite advertencias cuando el event loop se bloquea más tiempo del esperado. Es la herramienta para diagnosticar el anti-patrón de `time.sleep` dentro de funciones `async`.

**El problema a demostrar:**
```
time.sleep(0.3)       ← bloquea el hilo del OS durante 300ms
                         → el event loop no puede ejecutar NINGUNA otra coroutine
                         → gather con 3 tareas de 0.3s tarda 0.9s (secuencial)

await asyncio.sleep(0.3)  ← registra un callback y cede el control
                              → el event loop puede ejecutar otras coroutines
                              → gather con 3 tareas de 0.3s tarda ~0.3s (M4)
```

**Predicción:**
- `gather` con `asyncio.sleep(0.3)`: debería tardar **~0.3s**
- `gather` con `time.sleep(0.3)`: debería tardar **~0.9s** (sin mejora — M1)

El modo debug (`loop.set_debug(True)`) detectará el bloqueo y emitirá una advertencia en el segundo caso. Observa el output completo.

> Referencia: `04a_asyncio_fundamentos.md` — sección "time.sleep vs asyncio.sleep"

In [4]:
import asyncio
import time

# Habilitamos debug mode para ver bloqueos
loop = asyncio.get_event_loop()
loop.set_debug(True)

# Un umbral bajo para detectar bloqueos rápidamente
# (normalmente el umbral es 100ms)
loop.slow_callback_duration = 0.05  # 50ms

# Tarea bien escrita: libera el event loop
async def tarea_correcta(nombre: str):
    print(f'  {nombre}: inicio')
    await asyncio.sleep(0.3)   # wait(τ) — event loop libre
    print(f'  {nombre}: fin')

# Tarea mal escrita: BLOQUEA el event loop
async def tarea_bloqueante(nombre: str):
    print(f'  {nombre}: inicio')
    time.sleep(0.3)            # ← bloquea el hilo del OS entero
    print(f'  {nombre}: fin')

# ¿Qué diferencia ves en la salida?
print('=== gather con tareas CORRECTAS (asyncio.sleep) ===')
t0 = time.perf_counter()
await asyncio.gather(tarea_correcta('A'), tarea_correcta('B'), tarea_correcta('C'))
print(f'Tiempo: {time.perf_counter()-t0:.2f}s  (esperado: ~0.3s)\n')

print('=== gather con tareas BLOQUEANTES (time.sleep) ===')
t0 = time.perf_counter()
await asyncio.gather(tarea_bloqueante('X'), tarea_bloqueante('Y'), tarea_bloqueante('Z'))
print(f'Tiempo: {time.perf_counter()-t0:.2f}s  (esperado: ~0.9s — sin mejora)')
print()
print('Observa: con time.sleep, gather NO ayuda.')
print('time.sleep bloquea el event loop → ninguna otra coroutine puede avanzar.')

=== gather con tareas CORRECTAS (asyncio.sleep) ===
  A: inicio
  B: inicio
  C: inicio
  A: fin
  B: fin
  C: fin
Tiempo: 0.33s  (esperado: ~0.3s)

=== gather con tareas BLOQUEANTES (time.sleep) ===
  X: inicio
  X: fin
  Y: inicio
  Y: fin
  Z: inicio
  Z: fin
Tiempo: 0.91s  (esperado: ~0.9s — sin mejora)

Observa: con time.sleep, gather NO ayuda.
time.sleep bloquea el event loop → ninguna otra coroutine puede avanzar.


In [5]:
# Desactivar debug mode para el resto del notebook
loop.set_debug(False)

---

## Sección 3: Implementar M2 y M3 — por qué NO mejoran

**TAREA — implementación guiada**

Esta sección te pide implementar dos modelos *incorrectos* para CPU-bound e ineficientes para sus casos de uso, y medir por qué fallan la promesa de concurrencia/paralelismo.

**TAREA 3.1 — M2: async con await secuencial**

Ya viste M2 en la Sección 1. Ahora explícalo formalmente:
- ¿Qué condición de M4 (`exec(τⱼ) ∩ wait(τᵢ) ≠ ∅`) falla en M2?
- ¿En qué se diferencia `await fn1(); await fn2()` de `asyncio.gather(fn1(), fn2())`?

Escribe la respuesta como comentario en la celda antes de correr el código.

**TAREA 3.2 — M3: threading CPU-bound**

El GIL (Global Interpreter Lock) garantiza que solo un hilo ejecuta bytecode Python a la vez. Para tareas CPU-bound (`wait(τᵢ) = ∅`), el GIL nunca se libera — el threading no puede producir paralelismo real.

**Predicción:** con N=4 hilos en una tarea CPU-bound, el speedup esperado es **≈1×** (sin mejora). Puede incluso ser < 1× por el overhead de sincronización del GIL.

Implementa el threading y mide. ¿Coincide con la predicción?

In [6]:
# TAREA 3.1 — M2: async con await secuencial
# ¿Por qué M2 es idéntico a M1 en términos de tiempo?
#
# Respuesta formal: M2 viola la condición  exec(τⱼ) ∩ wait(τᵢ) ≠ ∅  que define M4.
#   - Con "await fn1(); await fn2()" el event loop espera que fn1 COMPLETE
#     antes de siquiera CREAR la coroutine fn2.
#   - En ningún momento hay dos tareas avanzando simultáneamente.
#   - T_total = N × T_tarea  ← igual que M1 (secuencial bloqueante).
#
# asyncio.gather(fn1(), fn2()) crea AMBAS coroutines al mismo tiempo:
#   - cuando fn1 hace "await asyncio.sleep(...)", el event loop pasa a fn2.
#   - La espera de fn1 y la espera de fn2 se solapan → T_total ≈ T_tarea_max.

# ── TAREA 3.2 — M3: threading CPU-bound ──────────────────────────────────────
def tarea_cpu_bound(n: int) -> int:
    """Tarea CPU-bound pura: wait(τᵢ) = ∅"""
    return sum(range(n))

N_CPU   = 30_000_000
N_HILOS = 4

# M1 secuencial (referencia)
t0 = time.perf_counter()
for _ in range(N_HILOS):
    tarea_cpu_bound(N_CPU)
t_secuencial = time.perf_counter() - t0

# M3: threading — el GIL impide paralelismo real en CPU-bound
t0 = time.perf_counter()
hilos = [threading.Thread(target=tarea_cpu_bound, args=(N_CPU,))
         for _ in range(N_HILOS)]
for h in hilos: h.start()
for h in hilos: h.join()
t_threading = time.perf_counter() - t0

speedup = t_secuencial / t_threading

print(f"M1 secuencial ({N_HILOS} tareas en serie): {t_secuencial:.2f}s")
print(f"M3 threading  ({N_HILOS} hilos, misma carga): {t_threading:.2f}s")
print(f"Speedup observado: {speedup:.2f}×  (esperado teórico sin GIL: {N_HILOS}×)")
print()
print("Conclusión:")
print("  El GIL garantiza que solo 1 hilo ejecuta bytecode Python a la vez.")
print("  Para tareas CPU-bound puras (wait=∅), el GIL NUNCA se libera.")
print("  Threading añade overhead de sincronización → puede ser más lento que M1.")
print("  Solución para CPU-bound: ProcessPoolExecutor (cada proceso tiene su propio GIL).")


M1 secuencial (4 tareas en serie): 1.75s
M3 threading  (4 hilos, misma carga): 1.68s
Speedup observado: 1.04×  (esperado teórico sin GIL: 4×)

Conclusión:
  El GIL garantiza que solo 1 hilo ejecuta bytecode Python a la vez.
  Para tareas CPU-bound puras (wait=∅), el GIL NUNCA se libera.
  Threading añade overhead de sincronización → puede ser más lento que M1.
  Solución para CPU-bound: ProcessPoolExecutor (cada proceso tiene su propio GIL).


## Sección 4: Race condition reproducible + fix con Lock

**TAREA — reproducir y corregir**

Las condiciones de carrera (race conditions) son el bug clásico de la concurrencia con memoria compartida. Ocurren cuando múltiples hilos leen y escriben la misma variable sin coordinación.

**Por qué ocurre:**
La operación `contador += 1` parece atómica pero en realidad son 3 pasos:
```
LOAD  contador      → lee el valor actual al registro de la CPU
ADD   registro, 1   → incrementa en 1
STORE registro → contador  → escribe de vuelta
```
Si dos hilos ejecutan esto concurrentemente, el segundo puede leer el valor *antes* de que el primero escriba su resultado — se pierde un incremento.

**Predicción:**
- Sin lock: `contador` final < `N_INCREMENTOS × N_HILOS` (incrementos perdidos, no determinístico)
- Con `threading.Lock`: `contador` final = `N_INCREMENTOS × N_HILOS` siempre

Corre varias veces sin lock. ¿El resultado es siempre distinto? ¿Siempre menor que el esperado?

> Nota: el GIL de Python reduce (no elimina) las race conditions. Este ejemplo las reproduce porque el increment no es atómico incluso con el GIL.

In [7]:
import threading

# TAREA 4.1 — Reproduce la race condition
N_INCREMENTOS = 100_000
N_HILOS_RACE  = 4

# Sin lock — resultado no determinista
contador_sin_lock = [0]

def incrementar_sin_lock():
    for _ in range(N_INCREMENTOS):
        contador_sin_lock[0] += 1   # NO atómico: LOAD, ADD, STORE separados

hilos = [threading.Thread(target=incrementar_sin_lock) for _ in range(N_HILOS_RACE)]
for h in hilos: h.start()
for h in hilos: h.join()

esperado = N_INCREMENTOS * N_HILOS_RACE
print(f"Sin lock  — esperado: {esperado:,}, obtenido: {contador_sin_lock[0]:,}")
print(f"Diferencia: {esperado - contador_sin_lock[0]:,} incrementos perdidos")
print()

# TAREA 4.2 — Fix con Lock
# Un Lock garantiza que la sección crítica (LOAD→ADD→STORE) es atómica:
# solo UN hilo puede estar dentro del bloque "with lock" en cualquier momento.

contador_con_lock = [0]
lock = threading.Lock()

def incrementar_con_lock():
    for _ in range(N_INCREMENTOS):
        with lock:
            contador_con_lock[0] += 1  # atómico: Lock protege LOAD+ADD+STORE

hilos_lock = [threading.Thread(target=incrementar_con_lock) for _ in range(N_HILOS_RACE)]
for h in hilos_lock: h.start()
for h in hilos_lock: h.join()

print(f"Con lock  — esperado: {esperado:,}, obtenido: {contador_con_lock[0]:,}")
print(f"Diferencia: {esperado - contador_con_lock[0]:,} incrementos perdidos")
assert contador_con_lock[0] == esperado, "¡El Lock no funcionó!"
print("✓ Con threading.Lock el resultado es siempre exactamente el esperado.")
print()
print("Nota de rendimiento:")
print("  Lock garantiza correctitud, pero serializa los hilos en la sección crítica.")
print("  Si el bloque 'with lock' es muy grande, el beneficio del threading desaparece.")
print("  Alternativa para contadores: threading.Semaphore o colecciones atómicas.")


Sin lock  — esperado: 400,000, obtenido: 400,000
Diferencia: 0 incrementos perdidos

Con lock  — esperado: 400,000, obtenido: 400,000
Diferencia: 0 incrementos perdidos
✓ Con threading.Lock el resultado es siempre exactamente el esperado.

Nota de rendimiento:
  Lock garantiza correctitud, pero serializa los hilos en la sección crítica.
  Si el bloque 'with lock' es muy grande, el beneficio del threading desaparece.
  Alternativa para contadores: threading.Semaphore o colecciones atómicas.


## Sección 5: Chatbot v2 con asyncio — N usuarios concurrentes

**TAREA — implementación completa**

Implementa el servidor chatbot v2 del Escenario A: LLM como API remota, I/O-bound. Compara la versión secuencial (v1) con la concurrente (v2).

**Arquitectura del chatbot v2:**
```
N usuarios simultáneos
        │
asyncio.gather(handle_request(0), handle_request(1), ..., handle_request(N-1))
        │
  ┌─────┴──────────────────────────────────────────┐
  │  Event loop (1 hilo)                           │
  │                                                │
  │  τ_u0 exec → await BD(50ms) → await LLM(1.5s) │
  │    τ_u1 exec → await BD(50ms) → await LLM(1.5s)│
  │      τ_u2 exec → await BD(50ms) → await LLM...│
  └──────────────────┬─────────────────────────────┘
                     │ (simultáneamente)
            [BD asyncpg]  [LLM API aiohttp]
```

**Predicciones:**
- v1 (secuencial) con N=10: `T_total ≈ 10 × 1.55s = 15.5s`
- v2 (gather) con N=10: `T_total ≈ 1.55s`
- Latencia del usuario 10 en v2: **similar a la del usuario 1** (todos esperan en paralelo)

Implementa `servidor_v1` y `servidor_v2`, mide con N=10, y responde:
1. ¿La latencia es uniforme entre usuarios en v2? ¿Por qué?
2. ¿Qué pasaría si uno de los usuarios tuviera `time.sleep` en su handler?

> Referencia: `04a_asyncio_fundamentos.md` — sección "Chatbot v2"

In [8]:
import asyncio
import time
import random

# Operaciones I/O-bound del chatbot (simuladas)
async def consultar_bd(user_id: int) -> list:
    """wait(τᵢ): I/O a base de datos — ~50ms"""
    await asyncio.sleep(0.05)
    return [f'historial de usuario {user_id}']

async def llamar_llm(historial: list) -> str:
    """wait(τᵢ): I/O a API del LLM — 1–2s variable"""
    await asyncio.sleep(random.uniform(1.0, 2.0))
    return f'respuesta para: {historial[-1]}'

async def handle_request(user_id: int) -> dict:
    """Una petición completa del chatbot v2 (M4)"""
    t_inicio = time.perf_counter()
    historial = await consultar_bd(user_id)
    respuesta = await llamar_llm(historial)
    latencia  = time.perf_counter() - t_inicio
    return {'user': user_id, 'respuesta': respuesta, 'latencia': latencia}

# ── TAREA 5.1 — Servidor secuencial (chatbot v1) ─────────────────────────────
async def servidor_v1(n_usuarios: int):
    """M1: atiende un usuario a la vez (await secuencial)."""
    resultados = []
    for i in range(n_usuarios):
        r = await handle_request(i)    # espera a que este termine antes del siguiente
        resultados.append(r)
    return resultados

# ── TAREA 5.2 — Servidor concurrente (chatbot v2) ────────────────────────────
async def servidor_v2(n_usuarios: int):
    """M4: atiende todos los usuarios concurrentemente con gather."""
    resultados = await asyncio.gather(
        *[handle_request(i) for i in range(n_usuarios)]
    )
    return list(resultados)

# ── TAREA 5.3 — Comparación v1 vs v2 con N=10 ────────────────────────────────
N = 10
print(f"Comparando v1 vs v2 con {N} usuarios...\n")

# v1 secuencial
t0 = time.perf_counter()
resultados_v1 = await servidor_v1(N)
t_v1 = time.perf_counter() - t0
lat_v1 = [r['latencia'] for r in resultados_v1]

print(f"=== v1 (secuencial) ===")
print(f"Tiempo total    : {t_v1:.2f}s  (esperado ≈ {N} × 1.55s = {N*1.55:.1f}s)")
print(f"Latencia prom.  : {sum(lat_v1)/len(lat_v1):.2f}s  |  "
      f"Latencia del usuario 10: {lat_v1[-1]:.2f}s")
print()

# v2 concurrente
t0 = time.perf_counter()
resultados_v2 = await servidor_v2(N)
t_v2 = time.perf_counter() - t0
lat_v2 = [r['latencia'] for r in resultados_v2]

print(f"=== v2 (gather — M4) ===")
print(f"Tiempo total    : {t_v2:.2f}s  (esperado ≈ 1.55s — el más lento define el total)")
print(f"Latencia prom.  : {sum(lat_v2)/len(lat_v2):.2f}s  |  "
      f"Latencia del usuario 10: {lat_v2[-1]:.2f}s")
print()
print(f"Speedup v2/v1   : {t_v1/t_v2:.1f}×")
print()
print("Respuesta 1 — ¿Latencia uniforme en v2?")
print("  SÍ. Todos los usuarios inician sus esperas al mismo tiempo (asyncio.gather).")
print("  Sus tiempos de I/O se solapan → la latencia de cada usuario ≈ su propio T_io + T_llm,")
print("  no la suma de todos los anteriores.")
print()
print("Respuesta 2 — ¿Qué pasa si un handler tiene time.sleep?")
print("  time.sleep bloquea el ÚNICO hilo del event loop.")
print("  Mientras ese handler duerme, ningún otro usuario puede avanzar.")
print("  → El beneficio de v2 desaparece: se comporta como v1 durante el bloqueo.")
print("  Fix: siempre usar 'await asyncio.sleep' (u otras corutinas async) en handlers async.")


Comparando v1 vs v2 con 10 usuarios...

=== v1 (secuencial) ===
Tiempo total    : 15.11s  (esperado ≈ 10 × 1.55s = 15.5s)
Latencia prom.  : 1.51s  |  Latencia del usuario 10: 1.08s

=== v2 (gather — M4) ===
Tiempo total    : 2.05s  (esperado ≈ 1.55s — el más lento define el total)
Latencia prom.  : 1.58s  |  Latencia del usuario 10: 1.84s

Speedup v2/v1   : 7.4×

Respuesta 1 — ¿Latencia uniforme en v2?
  SÍ. Todos los usuarios inician sus esperas al mismo tiempo (asyncio.gather).
  Sus tiempos de I/O se solapan → la latencia de cada usuario ≈ su propio T_io + T_llm,
  no la suma de todos los anteriores.

Respuesta 2 — ¿Qué pasa si un handler tiene time.sleep?
  time.sleep bloquea el ÚNICO hilo del event loop.
  Mientras ese handler duerme, ningún otro usuario puede avanzar.
  → El beneficio de v2 desaparece: se comporta como v1 durante el bloqueo.
  Fix: siempre usar 'await asyncio.sleep' (u otras corutinas async) en handlers async.
